In [ ]:
from typing import NamedTuple, Self, Iterable, Iterator
from itertools import chain, combinations, product

iterflat = chain.from_iterable


In [ ]:
from board import Loc, Node
from utils import nonone

In [ ]:
from types import EllipsisType

Every = EllipsisType
EVERY = Ellipsis

In [ ]:
# match zone.blk, zone.row, zone.col:
#     case Every(), Every(), Every():
#         pass
#     case int(b), Every(), Every():
#         pass
#     case Every(), int(r), Every():
#         pass
#     case Every(), Every(), int(c):
#         pass
#     case int(b), int(r), Every():
#         pass
#     case int(b), Every(), int(c):
#         pass
#     case int(b), int(r), int(c):
#         pass

In [ ]:
class Topo:
    """Relations between blocks/rows/cols"""

    IDX = (1, 2, 3, 4, 5, 6, 7, 8, 9)

    blks = IDX
    rows = IDX
    cols = IDX

    @staticmethod
    def blk4loc(row: int, col: int) -> int:
        assert 1 <= row <= 9 and 1 <= col <= 9
        r0 = (row - 1) // 3 * 3
        c0 = (col - 1) // 3
        return r0 + c0 + 1

    @staticmethod
    def blk4row(row: int) -> tuple[int, int, int]:
        assert 1 <= row <= 9
        b0 = (row - 1) // 3 * 3
        return (b0 + 1, b0 + 2, b0 + 3)

    @staticmethod
    def blk4col(col: int) -> tuple[int, int, int]:
        assert 1 <= col <= 9
        b0 = (col - 1) // 3
        return (b0 + 1, b0 + 4, b0 + 7)

    @staticmethod
    def row4blk(blk: int) -> tuple[int, int, int]:
        assert 1 <= blk <= 9
        r0 = (blk - 1) // 3 * 3
        return (r0 + 1, r0 + 2, r0 + 3)

    @staticmethod
    def col4blk(blk: int) -> tuple[int, int, int]:
        assert 1 <= blk <= 9
        c0 = (blk - 1) % 3 * 3
        return (c0 + 1, c0 + 2, c0 + 3)

    @staticmethod
    def valid(blk: int | Every, row: int | Every, col: int | Every) -> bool:
        # hard constraints
        assert blk is EVERY or 1 <= blk <= 9  # type: ignore
        assert row is EVERY or 1 <= row <= 9  # type: ignore
        assert col is EVERY or 1 <= col <= 9  # type: ignore

        # soft constraints
        try:
            if isinstance(blk, int):
                if isinstance(row, int):
                    assert row in Topo.row4blk(blk)
                if isinstance(col, int):
                    assert col in Topo.col4blk(blk)
            else:
                assert row is EVERY or col is EVERY
            return True
        except AssertionError:
            return False

In [ ]:
class Zone(NamedTuple):
    blk: int | EllipsisType
    row: int | EllipsisType
    col: int | EllipsisType

    @classmethod
    def B(cls, blk: int) -> Self:
        assert 1 <= blk <= 9
        return cls(blk, ..., ...)

    @classmethod
    def R(cls, row: int) -> Self:
        assert 1 <= row <= 9
        return cls(..., row, ...)

    @classmethod
    def C(cls, col: int) -> Self:
        assert 1 <= col <= 9
        return cls(..., ..., col)

    @classmethod
    def L(cls, row: int, col: int) -> Self:
        assert 1 <= row <= 9 and 1 <= col <= 9
        return cls(Topo.blk4loc(row, col), row, col)

    @classmethod
    def Allblk(cls) -> Iterable[Self]:
        return (cls(b, ..., ...) for b in Topo.blks)

    @classmethod
    def Allrow(cls) -> Iterable[Self]:
        return (cls(..., r, ...) for r in Topo.rows)

    @classmethod
    def Allcol(cls) -> Iterable[Self]:
        return (cls(..., ..., c) for c in Topo.cols)

    # invalid blk may occur because of all the mess around

    def valid(self) -> bool:
        return Topo.valid(self.blk, self.row, self.col)

    @property
    def is_everything(self):
        # it may appear in some calculations
        return self.blk is EVERY and self.row is EVERY and self.col is EVERY

    @property
    def is_cellular(self):
        return self.row is not EVERY and self.col is not EVERY

    @property
    def is_major(self):
        return sum(a is not EVERY for a in (self.blk, self.row, self.col)) == 1

    @classmethod
    def of(cls, what: Node | Loc | Self):
        if isinstance(what, cls):
            return what
        if isinstance(what, Loc):
            return cls(Topo.blk4loc(what.row, what.col), what.row, what.col)
        if isinstance(what, Node):
            loc = what.loc
            return cls(Topo.blk4loc(loc.row, loc.col), loc.row, loc.col)
        raise ValueError()

    def loc(self) -> Loc:
        assert isinstance(self.row, int) and isinstance(self.col, int)
        return Loc(self.row, self.col)

    @classmethod
    def intersection(cls, zone1: Self, zone2: Self) -> Self | None:
        """Intersection of zones
        == visible from both zones
        """
        if zone1.is_everything:  # WHY?
            return zone2
        if zone2.is_everything:  # WHY?
            return zone1

        assert zone1.valid() and zone2.valid()

        if zone1 == zone2:
            return zone1

        def eqe(cell, zone):
            # if a cellular zone matches some another zone
            return (zone.blk == EVERY or zone.blk == cell.blk) and (zone.row == EVERY or zone.row == cell.row) and (zone.col is EVERY or zone.col == cell.col)

        if zone1.is_cellular:
            return zone1 if eqe(zone1, zone2) else None
        if zone2.is_cellular:
            return zone2 if eqe(zone2, zone1) else None

        def validated(b, r, c):
            return cls(b, r, c) if Topo.valid(b, r, c) else None

        # TODO: optimize the shit
        match zone1.blk, zone1.row, zone1.col, zone2.blk, zone2.row, zone2.col:
            case Every(), int(r1), Every(), Every(), Every(), int(c2):  # R & C
                return cls(Topo.blk4loc(r1, c2), r1, c2)
            case Every(), Every(), int(c1), Every(), int(r2), Every():  # C & R
                return cls(Topo.blk4loc(r2, c1), r2, c1)
            case int(b1), Every(), Every(), Every(), int(r2), Every():  # B & R
                return validated(b1, r2, ...)
            case int(b1), Every(), Every(), Every(), Every(), int(c2):  # B & C
                return validated(b1, ..., c2)
            case Every(), int(r1), Every(), int(b2), Every(), Every():  # R & B
                return validated(b2, r1, ...)
            case Every(), Every(), int(c1), int(b2), Every(), Every():  # C & B
                return validated(b2, ..., c1)
            # overlapping cases
            case int(b1), int(r1), Every(), int(b2), Every(), Every():  # BR & B
                return zone1 if b1 == b2 else None
            case int(b1), int(r1), Every(), Every(), int(r2), Every():  # BR & R
                return zone1 if r1 == r2 else None
            case int(b1), Every(), int(c1), int(b2), Every(), Every():  # BC & B
                return zone1 if b1 == b2 else None
            case int(b1), Every(), int(c1), Every(), Every(), int(c2):  # BC & C
                return zone1 if c1 == c2 else None
            case int(b1), Every(), Every(), int(b2), int(r2), Every():  # B & BR
                return zone2 if b1 == b2 else None
            case Every(), int(r1), Every(), int(b2), int(r2), Every():  # R & BR
                return zone2 if r1 == r2 else None
            case int(b1), Every(), Every(), int(b2), Every(), int(c2):  # B & BC
                return zone2 if b1 == b2 else None
            case Every(), Every(), int(c1), int(b2), Every(), int(c2):  # C & BC
                return zone2 if c1 == c2 else None
            # cellularizing
            case int(b1), int(r1), Every(), Every(), Every(), int(c2):  # BR & C
                return validated(b1, r1, c2)
            case int(b1), Every(), int(c1), Every(), int(r2), Every():  # BC & R
                return validated(b1, r2, c1)
            case int(b1), int(r1), Every(), int(b2), Every(), int(c2):  # BR & BC
                return validated(b1, r1, c2) if b1 == b2 else None
            case Every(), Every(), int(c1), int(b2), int(r2), Every():  # C & BR
                return validated(b2, r2, c1)
            case Every(), int(r1), Every(), int(b2), Every(), int(c2):  # BC & R
                return validated(b2, r1, c2)
            case int(b1), Every(), int(c1), int(b2), int(r2), Every():  # BC & BR
                return validated(b2, r2, c1) if b1 == b2 else None
            case _:
                return None

    def issubzone(self, other: Self):
        """This zone fully contained in another (or equal)
        = fully mutually visible
        """

        if self == other:
            return True

        if other.is_everything:
            return True

        if self.is_major:
            return self == other

        match self.blk, self.row, self.col, other.blk, other.row, other.col:
            case int(b1), _, _, int(b2), Every(), Every():
                return b1 == b2
            case _, int(r1), _, Every(), int(r2), Every():
                return r1 == r2
            case _, _, int(c1), Every(), Every(), int(c2):
                return c1 == c2

        return False

    @classmethod
    def around(cls, zone: Self) -> Iterable[Self]:
        """All major zones fully containing a subzone
        = visible by any cell in the subzone
        """
        assert zone.valid()

        match zone.blk, zone.row, zone.col:
            case int(b), int(r), Every():
                yield cls(b, ..., ...)
                yield cls(..., r, ...)
            case int(b), Every(), int(c):
                yield cls(b, ..., ...)
                yield cls(..., ..., c)
            case int(b), int(r), int(c):
                yield cls(b, ..., ...)
                yield cls(..., r, ...)
                yield cls(..., ..., c)

    @classmethod
    def across(cls, zone: Self):
        """All major zones intersecting given
        = vizible by some cells in the zone
        """
        assert zone.valid()

        match zone.blk, zone.row, zone.col:
            case int(b), Every(), Every():
                yield from (cls(..., r, ...) for r in Topo.row4blk(b))
                yield from (cls(..., ..., c) for c in Topo.col4blk(b))
            case Every(), int(r), Every():
                yield from (cls(b, ..., ...) for b in Topo.blk4row(r))
                yield from (cls(..., ..., c) for c in Topo.cols)
            case Every(), Every(), int(c):
                yield from (cls(b, ..., ...) for b in Topo.blk4col(c))
                yield from (cls(..., r, ...) for r in Topo.rows)
            case int(b), int(r), Every():
                yield from (cls(..., ..., c) for c in Topo.col4blk(b))
            case int(b), Every(), int(c):
                yield from (cls(..., r, ...) for r in Topo.row4blk(b))

    @classmethod
    def partitions(cls, majzone: Self) -> Iterable[Self]:
        """All subzones of a major zone, partitioned by intersections with others"""
        assert majzone.valid() and majzone.is_major

        # quick stuff without nested iterations and redundant overlappency
        match majzone.blk, majzone.row, majzone.col:
            case int(b), Every(), Every():
                yield from (cls(b, r, ...) for r in Topo.row4blk(b))
                yield from (cls(b, ..., c) for c in Topo.col4blk(b))
            case Every(), int(r), Every():
                yield from (cls(b, r, ...) for b in Topo.blk4row(r))
            case Every(), Every(), int(c):
                yield from (cls(b, ..., c) for b in Topo.blk4col(c))

    def __iter__(self) -> Iterator[Loc]:
        """Iterate all cell locations in the zone"""

        assert self.valid()

        match self.blk, self.row, self.col:
            case Every(), Every(), Every():  # WHY ?
                yield from (Loc(r, c) for r in Topo.rows for c in Topo.cols)
            case _, int(r), int(c):
                yield Loc(r, c)
            case Every(), int(r), Every():
                yield from (Loc(r, c) for c in Topo.cols)
            case Every(), Every(), int(c):
                yield from (Loc(r, c) for r in Topo.rows)
            case int(b), Every(), Every():
                yield from (Loc(r, c) for r in Topo.row4blk(b) for c in Topo.col4blk(b))
            case int(b), int(r), Every():
                yield from (Loc(r, c) for c in Topo.col4blk(b))
            case int(b), Every(), int(c):
                yield from (Loc(r, c) for r in Topo.row4blk(b))

    def __contains__(self, loc: Loc) -> bool:
        """If the zone contains specific location"""

        match self.blk, self.row, self.col:
            case Every(), Every(), Every():
                return True
            case int(b), Every(), Every():
                return loc.row in Topo.row4blk(b) and loc.col in Topo.col4blk(b)
            case Every(), int(r), Every():
                return loc.row == r
            case Every(), Every(), int(c):
                return loc.col == c
            case int(b), int(r), Every():
                return loc.row == r and loc.col in Topo.col4blk(b)
            case int(b), Every(), int(c):
                return loc.col == c and loc.row in Topo.row4blk(b)
            case _, int(r), int(c):
                return loc.row == r and loc.col == c

    def __le__(self, other: Self):
        return self.issubzone(other)

    def __lt__(self, other: Self):
        return self.issubzone(other) and self != other

    def __ge__(self, other: Self):
        return other.issubzone(self)

    def __gt__(self, other: Self):
        return other.issubzone(self) and self != other

    def __and__(self, other: Self):
        return self.__class__.intersection(self, other)

    def __str__(self):
        rstr = "…" if self.row is EVERY else f"r{self.row}"
        cstr = "…" if self.col is EVERY else f"c{self.col}"
        if self.is_cellular:
            return f"[{rstr}{cstr}]"
        else:
            bstr = "…" if self.blk is EVERY else f"b{self.blk}"
            return f"[{bstr}{rstr}{cstr}]"

In [ ]:
assert set(iter(Zone(5, ..., ...))) == {
    Loc(4, 4),
    Loc(4, 5),
    Loc(4, 6),
    Loc(5, 4),
    Loc(5, 5),
    Loc(5, 6),
    Loc(6, 4),
    Loc(6, 5),
    Loc(6, 6),
}

In [ ]:
assert set(iter(Zone(..., 6, ...))) == {
    Loc(6, 1),
    Loc(6, 2),
    Loc(6, 3),
    Loc(6, 4),
    Loc(6, 5),
    Loc(6, 6),
    Loc(6, 7),
    Loc(6, 8),
    Loc(6, 9),
}

In [ ]:
assert set(iter(Zone(5, 6, ...))) == {
    Loc(6, 4),
    Loc(6, 5),
    Loc(6, 6),
}

In [ ]:
assert Loc(6, 4) in Zone(5, ..., ...)
assert Loc(6, 4) in Zone(..., 6, ...)
assert Loc(6, 4) in Zone(..., ..., 4)
assert Loc(6, 4) in Zone(5, 6, ...)
assert Loc(6, 4) in Zone(5, ..., 4)

In [ ]:
assert Zone(5, ..., ...) & Zone(..., 6, ...) == Zone(5, 6, ...)
assert Zone(5, ..., ...) & Zone(..., 7, ...) is None

In [ ]:
assert Zone(5, ..., ...) >= Zone(5, 6, ...)
assert Zone(5, 6, ...) <= Zone(5, ..., ...)
assert Zone(5, 6, ...) <= Zone(..., 6, ...)

In [ ]:
assert set(Zone.around(Zone(5, ..., ...))) == set()
assert set(Zone.around(Zone(5, 6, ...))) == {
    Zone(5, ..., ...),
    Zone(..., 6, ...),
}
assert set(Zone.around(Zone(5, 6, 4))) == {
    Zone(5, ..., ...),
    Zone(..., 6, ...),
    Zone(..., ..., 4),
}

In [ ]:
assert set(Zone.across(Zone(5, ..., ...))) == {
    Zone(..., 4, ...),
    Zone(..., 5, ...),
    Zone(..., 6, ...),
    Zone(..., ..., 4),
    Zone(..., ..., 5),
    Zone(..., ..., 6),
}
assert set(Zone.across(Zone(5, 6, ...))) == {
    Zone(..., ..., 4),
    Zone(..., ..., 5),
    Zone(..., ..., 6),
}

In [ ]:
assert set(Zone.partitions(Zone(5, ..., ...))) == {
    Zone(5, 4, ...),
    Zone(5, 5, ...),
    Zone(5, 6, ...),
    Zone(5, ..., 4),
    Zone(5, ..., 5),
    Zone(5, ..., 6),
}
assert set(Zone.partitions(Zone(..., 6, ...))) == {
    Zone(4, 6, ...),
    Zone(5, 6, ...),
    Zone(6, 6, ...),
}
assert set(Zone.partitions(Zone(..., ..., 7))) == {
    Zone(3, ..., 7),
    Zone(6, ..., 7),
    Zone(9, ..., 7),
}

In [ ]:
def allvisible(z1: Zone, z2: Zone) -> set[Zone]:
    """All zones fully visible from both of the observers
    The zones may overlap
    """
    intervis = set(z1a & z2a for z1a, z2a in product(Zone.around(z1), Zone.around(z2)))
    intervis -= {None}
    return intervis  # type: ignore


In [ ]:
assert allvisible(Zone.L(2, 3), Zone.L(7, 8)) == {
    Zone(3, 2, 8),
    Zone(7, 7, 3),
}

assert allvisible(Zone(1, 2, ...), Zone(2, 3, ...)) == {
    Zone(1, 3, ...),
    Zone(2, 2, ...),
}


In [ ]:
def visibility(z1: Zone, z2: Zone):
    """Check if the zones are fully mutually visible
    Return major zones through which they see each other
    == common major zones
    """
    return set(Zone.around(z1)) & set(Zone.around(z2))

In [ ]:
assert visibility(Zone(1, 2, 3), Zone(3, 2, 9)) == {Zone(..., 2, ...)}
assert visibility(Zone(1, ..., 3), Zone(4, ..., 3)) == {Zone(..., ..., 3)}
assert visibility(Zone(1, ..., 3), Zone(4, 5, 3)) == {Zone(..., ..., 3)}

assert visibility(Zone(1, 2, 3), Zone(1, 3, 3)) == {Zone(1, ..., ...), Zone(..., ..., 3)}


In [ ]:
from canvas import SudokuCanvas, XY

canva = SudokuCanvas()

In [ ]:
from canvas import P0

cnv = canva[canva.HLIGHT_LAYER]
cnv.fill_style = "rgba(255, 255, 0, 0.5)"
cnv.line_width = 4
cnv.stroke_style = "rgba(240, 240, 240, 0.75)"


def bounds(zone: Zone) -> tuple[Loc, Loc]:
    locs = tuple(iter(zone))
    print(locs)
    rmin = min(l.row for l in locs)
    rmax = max(l.row for l in locs)
    cmin = min(l.col for l in locs)
    cmax = max(l.col for l in locs)

    return Loc(rmin, cmin), Loc(rmax, cmax)


def highlight(l1: Loc, l2: Loc):
    p1, p2 = XY.ofsect(l1, l2)
    p1 = XY.add(p1, P0)
    p2 = XY.add(P0, p2)
    cnv.fill_rect(p1.x, p1.y, p2.x - p1.x, p2.y - p1.y)


def outline(l1: Loc, l2: Loc):
    p1, p2 = XY.ofsect(l1, l2)
    p1 = XY.add(p1, P0)
    p2 = XY.add(P0, p2)
    cnv.stroke_rect(p1.x + 3, p1.y + 3, p2.x - p1.x - 6, p2.y - p1.y - 6)


def highlight_zone(zone: Zone):
    lmin, lmax = bounds(zone)
    highlight(lmin, lmax)


def outline_zone(zone: Zone):
    lmin, lmax = bounds(zone)
    outline(lmin, lmax)


In [ ]:
display(canva)

In [ ]:
zones = [
    Zone(1, ..., ...),
    Zone(5, ..., ...),
    Zone(..., 3, ...),
    Zone(..., 5, ...),
    Zone(..., ..., 3),
    Zone(..., ..., 5),
]

In [ ]:
it = iter(zones)

In [ ]:
cnv.clear()
z = next(it)
print(z)
highlight_zone(z)

In [ ]:
it = ((z1, z2, z1 & z2) for z1, z2 in combinations(zones, 2))


In [ ]:
cnv.clear()
z1, z2, zr = next(it)
print(z1, z2, "=>", zr)
highlight_zone(z1)
highlight_zone(z2)
if zr is not None:
    outline_zone(zr)

In [ ]:
# it = ((z, set(Zone.intersecting(z))) for z in zones)
it = ((z, set(Zone.partitions(z))) for z in zones)

In [ ]:
cnv.clear()
z1, zr = next(it)
print(z1, "=>", set(map(str, zr)))
highlight_zone(z1)
for z in zr:
    outline_zone(z)